# Mini-Project 2: Pre-paid card marketing (FINA 4390)
**Goal:** predict who uses a reloadable prepaid card not linked to a bank account (`PRODUSE_3`) with the CFPB National Financial Well-Being Survey, then answer two questions per stakeholder.

* **Bank (card issuer):** Q1 who to target (marketing and origination); Q2 key insights on usage likelihood.
* **CFPB (regulator):** Q3 can we identify at-risk groups; Q4 what policies protect them.

We split the project: each stakeholder gets its own feature set (up to 20 variables), the same three models (Logit, Random Forest, AdaBoost), 5-fold stratified cross-validation for tuning and model choice, and a held-out 20% test set for the final check. Survey weights are ignored, as instructed.

In [ ]:
FAST = False          # True = tiny grids for a quick test run (about a minute)
DATA_PATH = 'CFPB-financial-wellness-data.csv'   # the NFWBS public-use file (CSV)
SEED = 0
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.base import clone
pd.set_option('display.width', 160)

## 1. Data cleaning and feature engineering
Quality checks (duplicates, missing outcome), CFPB special codes set to missing (-1 refused, -2 not asked, -4 not written, 8/98 don't know, 99 prefer not to say), and row-wise feature engineering. Nothing here learns from the data, so it can safely run before the split; imputation and scaling are fit later, on training folds only.

In [ ]:
import pandas as pd, numpy as np

RAW = DATA_PATH

def load_and_clean(path=RAW):
    df = pd.read_csv(path)
    report = {}
    # ---------- 1. data quality checks ----------
    report['rows'], report['cols'] = df.shape
    report['dup_ids'] = int(df['PUF_ID'].duplicated().sum())
    report['dup_rows'] = int(df.drop(columns='PUF_ID').duplicated().sum())
    report['outcome_missing'] = int(df['PRODUSE_3'].isna().sum())
    # ---------- 2. CFPB special codes -> missing ----------
    # negative codes: -1 refused, -2 not asked, -4 not written to file, -5 county unknown
    # non-substantive answers: 8/98 don't know / not sure, 99 prefer not to say / refused
    nonsub = {'SAVINGSRANGES': [98, 99], 'ABSORBSHOCK': [8], 'COLLECT': [8],
              'SNAP': [8], 'FRAUD2': [8], 'EMPLOY': [99]}
    used = ['PPINCIMP','agecat','PPEDUC','EMPLOY','HOUSING','PPHHSIZE','PPREG4','PRODHAVE_1',
            'PRODUSE_1','PRODUSE_2','PRODUSE_4','PRODUSE_5','SAVINGSRANGES','REJECTED_1','REJECTED_2',
            'COLLECT','VOLATILITY','PRODHAVE_4','KHscore','LMscore','FSscore','FWBscore','ABSORBSHOCK',
            'ENDSMEET','VOLATILITY','CONSPROTECT1','CONSPROTECT2','SCFHORIZON','SNAP','SUBNUMERACY1',
            'MATHARDSHIP_1','MATHARDSHIP_2','MATHARDSHIP_3','MATHARDSHIP_4','MATHARDSHIP_5','MATHARDSHIP_6',
            'PPETHM','PPGENDER','PPMARIT','FRAUD2','PPMSACAT','EARNERS']
    used = list(dict.fromkeys(used))
    special_counts = {}
    for c in used:
        # KHscore is an IRT score whose real values can be negative, so only the other codes apply
        bad = (df[c] < 0) if c != 'KHscore' else pd.Series(False, index=df.index)
        if c in nonsub:
            bad |= df[c].isin(nonsub[c])
        special_counts[c] = int(bad.sum())
        df.loc[bad, c] = np.nan
    report['special_codes'] = special_counts
    # ---------- 3. feature engineering (row-wise, no learned statistics -> no leakage) ----------
    d = pd.DataFrame(index=df.index)
    d['income'] = df['PPINCIMP']                      # 1 (<$20k) ... 9 ($150k+)
    d['age_group'] = df['agecat']                     # 1 (18-24) ... 8 (75+)
    d['age_under35'] = (df['agecat'] <= 2).astype(float)   # usage is U-shaped in age,
    d['age_75plus'] = (df['agecat'] == 8).astype(float)    # so two dummies beat one linear term
    d['education'] = df['PPEDUC']                     # 1 (<HS) ... 5 (graduate)
    emp = df['EMPLOY']
    d['emp_part_time'] = (emp == 3).astype(float)
    d['emp_self_employed'] = (emp == 1).astype(float)
    d['emp_unemployed_or_unable'] = emp.isin([6, 7]).astype(float)
    d['emp_retired'] = (emp == 8).astype(float)
    d['emp_student_or_homemaker'] = emp.isin([4, 5]).astype(float)
    for col in ['emp_part_time','emp_self_employed','emp_unemployed_or_unable','emp_retired','emp_student_or_homemaker']:
        d.loc[emp.isna(), col] = np.nan
    d['renter'] = (df['HOUSING'] == 2).astype(float)
    d['housing_neither_own_nor_rent'] = (df['HOUSING'] == 3).astype(float)
    d.loc[df['HOUSING'].isna(), ['renter','housing_neither_own_nor_rent']] = np.nan
    d['household_size'] = df['PPHHSIZE']
    d['region_south'] = (df['PPREG4'] == 3).astype(float)
    d['region_midwest'] = (df['PPREG4'] == 2).astype(float)
    d['region_west'] = (df['PPREG4'] == 4).astype(float)
    d['unbanked'] = 1 - df['PRODHAVE_1']               # no checking/savings account
    d['used_payday_loan'] = df['PRODUSE_1']
    d['used_pawn_or_title_loan'] = df['PRODUSE_2']
    d['used_nonbank_money_transfer'] = df['PRODUSE_4']
    d['used_check_cashing'] = df['PRODUSE_5']
    d['alt_financial_services_count'] = df[['PRODUSE_1','PRODUSE_2','PRODUSE_4','PRODUSE_5']].sum(axis=1)
    d['savings_range'] = df['SAVINGSRANGES']           # 1 ($0) ... 7 ($75k+)
    d['savings_not_reported'] = d['savings_range'].isna().astype(float)   # 'incorporate' missingness
    d['turned_down_for_credit'] = df['REJECTED_1']
    d['credit_constrained'] = ((df['REJECTED_1'] == 1) | (df['REJECTED_2'] == 1)).astype(float)
    d.loc[df['REJECTED_1'].isna() & df['REJECTED_2'].isna(), 'credit_constrained'] = np.nan
    d['debt_collector_contact'] = df['COLLECT']
    d['income_volatility'] = df['VOLATILITY']          # 1 stable ... 3 often varies
    d['has_retirement_account'] = df['PRODHAVE_4']
    d['fin_knowledge_KH'] = df['KHscore']
    d['fin_literacy_LM'] = df['LMscore']               # 0-3 basic questions right
    d['fin_skill'] = df['FSscore']
    d['fin_wellbeing'] = df['FWBscore']
    d['can_raise_2000'] = df['ABSORBSHOCK']            # 1 certain could not ... 4 certain could
    d['shock_answer_dont_know'] = (df['ABSORBSHOCK'].isna()).astype(float)
    d['hard_to_pay_bills'] = df['ENDSMEET']            # 1 not difficult ... 3 very difficult
    mh = df[[f'MATHARDSHIP_{i}' for i in range(1, 7)]]
    d['hardship_count'] = (mh >= 2).sum(axis=1).astype(float)
    d.loc[mh.isna().all(axis=1), 'hardship_count'] = np.nan
    d['felt_mistreated_by_fin_firms'] = df['CONSPROTECT1']   # 1 never ... 4 often
    d['knows_where_to_complain'] = df['CONSPROTECT2']        # 1 not familiar ... 3 very
    d['planning_horizon'] = df['SCFHORIZON']                 # 1 next few months ... 5 10+ yrs
    d['receives_snap'] = df['SNAP']
    d['numeracy_self_rating'] = df['SUBNUMERACY1']
    d['metro'] = df['PPMSACAT']
    # kept OUT of every model (protected classes) but used for the fairness check
    prot = df[['PPETHM','PPGENDER','PPMARIT']].copy()
    y = df['PRODUSE_3'].astype(int)
    return d, y, prot, df, report

BANK_FEATURES = {
 'Household income': ['income'],
 'Age: under 35 / 75+': ['age_under35','age_75plus'],
 'Education': ['education'],
 'Employment status': ['emp_part_time','emp_self_employed','emp_unemployed_or_unable','emp_retired','emp_student_or_homemaker'],
 'Renter / neither own nor rent': ['renter','housing_neither_own_nor_rent'],
 'Household size': ['household_size'],
 'Region': ['region_south','region_midwest','region_west'],
 'No bank account': ['unbanked'],
 'Used payday loan': ['used_payday_loan'],
 'Used pawn / title loan': ['used_pawn_or_title_loan'],
 'Used non-bank money transfer': ['used_nonbank_money_transfer'],
 'Used check cashing / money orders': ['used_check_cashing'],
 'Savings (range) + not reported': ['savings_range','savings_not_reported'],
 'Turned down for credit': ['turned_down_for_credit'],
 'Debt collector contact': ['debt_collector_contact'],
 'Income volatility': ['income_volatility'],
 'Has retirement account': ['has_retirement_account'],
}
CFPB_FEATURES = {
 'Financial knowledge (Knoll-Houts)': ['fin_knowledge_KH'],
 'Basic literacy (3 questions)': ['fin_literacy_LM'],
 'Could raise $2,000 in 30 days': ['can_raise_2000','shock_answer_dont_know'],
 'Hard to pay bills': ['hard_to_pay_bills'],
 'Material hardships (count)': ['hardship_count'],
 'Income volatility': ['income_volatility'],
 'Savings (range) + not reported': ['savings_range','savings_not_reported'],
 'No bank account': ['unbanked'],
 'Other high-cost services used (count)': ['alt_financial_services_count'],
 'Debt collector contact': ['debt_collector_contact'],
 'Credit constrained': ['credit_constrained'],
 'Felt mistreated by financial firms': ['felt_mistreated_by_fin_firms'],
 'Knows where to complain': ['knows_where_to_complain'],
 'Planning horizon': ['planning_horizon'],
 'Receives SNAP': ['receives_snap'],
 'Not working (unemployed / unable to work)': ['emp_unemployed_or_unable'],
 'Household income': ['income'],
 'Age: under 35 / 75+': ['age_under35','age_75plus'],
 'Education': ['education'],
}
def cols(fs): return [c for v in fs.values() for c in v]

In [ ]:
d, y, prot, raw, report = load_and_clean()
print(f"{report['rows']:,} rows x {report['cols']} columns | duplicate IDs: {report['dup_ids']} | duplicate rows: {report['dup_rows']} | missing outcome: {report['outcome_missing']}")
print(f"Prepaid users (PRODUSE_3 = 1): {y.sum()} of {len(y):,} ({y.mean():.1%})")
print('Special codes set to missing:', {k: v for k, v in report['special_codes'].items() if v})
print(f"Savings not reported (kept as a flag): {d['savings_range'].isna().mean():.1%}")

## 2. Feature sets
**Bank:** what an issuer can observe or buy at application (no protected classes; SNAP excluded because public-assistance receipt is protected under ECOA).
**CFPB:** signs of vulnerability and low financial sophistication. Race/ethnicity, sex and marital status are kept out of both models and used only for a fairness check.

In [ ]:
def vif(X):
    Xi = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X), columns=X.columns)
    out = {}
    for c in Xi.columns:
        others = Xi.drop(columns=c)
        out[c] = 1 / (1 - LinearRegression().fit(others, Xi[c]).score(others, Xi[c]))
    return pd.Series(out).sort_values(ascending=False)
FS = {'BANK': BANK_FEATURES, 'CFPB': CFPB_FEATURES}
for name, fs in FS.items():
    print(f'{name}: {len(fs)} variables ({len(cols(fs))} model columns)')
    for v in fs: print('   -', v)
    print(f'   highest VIF: {vif(d[cols(fs)]).max():.2f}  (above 5 would signal harmful multicollinearity)\n')

## 3. Split, three models, cross-validated tuning
80/20 split stratified on the outcome. Each model is a pipeline (median imputation, standardization, model), so imputation and scaling are re-fit inside every training fold: no leakage. Balanced class weights handle the 8.8% minority class; AdaBoost re-weights misclassified cases by design. Grid search is scored by AUC with 5-fold stratified CV.

In [ ]:
idx_train, idx_test = train_test_split(d.index, test_size=0.2, stratify=y, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(f'train {len(idx_train):,} ({y[idx_train].sum()} users) | test {len(idx_test):,} ({y[idx_test].sum()} users)')
def make_models():
    return {
     'Logit': Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                        ('model', LogisticRegression(class_weight='balanced', max_iter=5000))]),
     'Random Forest': Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                        ('model', RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample', random_state=SEED, n_jobs=1))]),
     'AdaBoost': Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                        ('model', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), random_state=SEED))]),
    }
GRIDS = {
 'Logit': {'model__C': [0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]},
 'Random Forest': {'model__n_estimators': [300], 'model__max_depth': [4, 6, 8, 12],
                   'model__min_samples_leaf': [10, 25, 50], 'model__max_features': ['sqrt', 0.3]},
 'AdaBoost': {'model__n_estimators': [200, 500], 'model__learning_rate': [0.02, 0.05, 0.1, 0.3],
              'model__estimator__max_depth': [1, 2]},
}
if FAST:
    GRIDS = {'Logit': {'model__C': [0.001]},
             'Random Forest': {'model__n_estimators': [50], 'model__max_depth': [4], 'model__min_samples_leaf': [50], 'model__max_features': ['sqrt']},
             'AdaBoost': {'model__n_estimators': [50], 'model__learning_rate': [0.3], 'model__estimator__max_depth': [1]}}

In [ ]:
TUNED, rows = {}, []
for who, fs in FS.items():                     # takes several minutes with the full grids
    X = d[cols(fs)]
    TUNED[who] = {}
    for m, pipe in make_models().items():
        gs = GridSearchCV(pipe, GRIDS[m], scoring='roc_auc', cv=cv, n_jobs=1, refit=True)
        gs.fit(X.loc[idx_train], y.loc[idx_train])
        folds = [gs.cv_results_[f'split{k}_test_score'][gs.best_index_] for k in range(5)]
        TUNED[who][m] = dict(est=gs.best_estimator_, cv_auc=np.mean(folds), cv_sd=np.std(folds), folds=folds, best=gs.best_params_)
        rows.append([who, m, round(np.mean(folds), 3), round(np.std(folds), 3), [round(f, 3) for f in folds], gs.best_params_])
        print(who, m, 'done')
cv_table = pd.DataFrame(rows, columns=['stakeholder', 'model', 'CV AUC', 'CV SD', 'fold AUCs', 'best settings'])
cv_table

## 4. Pick the model by cross-validated AUC, then check it once on the test set
Each stakeholder flags a share of people: the bank markets to its top 20% of scores; the CFPB casts a wider net (top 30%) because missing a vulnerable user costs more than extra outreach. The score cutoff is set on out-of-fold training predictions, never on the test set.

In [ ]:
SHARE = {'BANK': 0.20, 'CFPB': 0.30}
yt = y.loc[idx_test]; rows = []; CHOSEN = {}
for who, fs in FS.items():
    X = d[cols(fs)]
    for m, t in TUNED[who].items():
        p = t['est'].predict_proba(X.loc[idx_test])[:, 1]
        oof = cross_val_predict(clone(t['est']), X.loc[idx_train], y.loc[idx_train], cv=cv, method='predict_proba')[:, 1]
        cut = np.quantile(oof, 1 - SHARE[who])
        tn, fp, fn, tp = confusion_matrix(yt, (p >= cut).astype(int)).ravel()
        t.update(cut=cut, cm=(tp, fp, fn, tn))
        rows.append([who, m, round(t['cv_auc'], 3), round(roc_auc_score(yt, p), 3), tp, fp, fn, tn,
                     f'{tp/(tp+fp):.1%}', f'{tp/(tp+fn):.1%}', f'{(tp+tn)/len(yt):.1%}'])
    CHOSEN[who] = max(TUNED[who], key=lambda m: TUNED[who][m]['cv_auc'])
test_table = pd.DataFrame(rows, columns=['stakeholder', 'model', 'CV AUC', 'test AUC', 'TP', 'FP', 'FN', 'TN', 'precision', 'recall', 'accuracy'])
print('Chosen models:', CHOSEN, '| naive "nobody uses prepaid" accuracy:', f'{1 - yt.mean():.1%}')
test_table

In [ ]:
WORDS = {'BANK': ('Customers reached', 'Wasted outreach', 'Customers missed', 'Correctly skipped'),
         'CFPB': ('At-risk users protected', 'Extra outreach', 'Users we miss', 'Correctly left alone')}
for who in FS:
    tp, fp, fn, tn = TUNED[who][CHOSEN[who]]['cm']; w = WORDS[who]
    print(f'\n{who} ({CHOSEN[who]}) on {len(yt):,} test people, {yt.sum()} users')
    print(f'   {w[0]}: {tp} | {w[2]}: {fn} | {w[1]}: {fp} | {w[3]}: {tn}')
    print(f'   flagged {tp+fp} ({(tp+fp)/len(yt):.0%}); caught {tp/(tp+fn):.0%} of users; '
          f'1 in {(tp+fp)/tp:.1f} flagged is a user vs 1 in {len(yt)/yt.sum():.1f} at random')

## 5. What drives usage: standardized logit coefficients (effect of a one-SD change, holding the rest fixed)

In [ ]:
for who in FS:
    coef = pd.Series(TUNED[who]['Logit']['est'].named_steps['model'].coef_[0], index=cols(FS[who]))
    print(f'\n{who}:'); print(coef.sort_values(key=abs, ascending=False).round(3).to_string())

## 6. Lift: prepaid use by score decile (cross-validated scores for all respondents, so no one is scored by a model trained on them)

In [ ]:
TIER = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, who in zip(axes, FS):
    X = d[cols(FS[who])]
    s = cross_val_predict(clone(TUNED[who][CHOSEN[who]]['est']), X, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method='predict_proba')[:, 1]
    rank = pd.Series(s, index=d.index).rank(method='first', ascending=False)
    dec = np.ceil(rank / len(s) * 10).astype(int)
    tab = y.groupby(dec).agg(rate='mean', users='sum', n='size'); tab['cumulative share of users'] = tab['users'].cumsum() / y.sum()
    TIER[who] = rank <= SHARE[who] * len(s)
    print(f'\n{who}: AUC of cross-validated scores {roc_auc_score(y, s):.3f}'); print(tab.round(3).to_string())
    ax.bar(tab.index, tab['rate'] * 100, color='#D9782A' if who == 'BANK' else '#1F4E8C')
    ax.axhline(y.mean() * 100, ls='--', c='k'); ax.set_xticks(range(1, 11))
    ax.set_title(f'{who}: prepaid use by score decile'); ax.set_xlabel('decile (1 = highest score)')
axes[0].set_ylabel('% using prepaid'); plt.tight_layout(); plt.show()

## 7. Insight tables for the four questions

In [ ]:
def compare(conds, tier):
    return pd.DataFrame({k: {'tier %': 100 * v.astype(float)[tier].mean(), 'everyone else %': 100 * v.astype(float)[~tier].mean()}
                         for k, v in conds.items()}).T.round(1)
nn = lambda s, c: c.where(s.notna())            # keep non-answers out of the percentages
print('Q1 BANK: top 20% of scores vs everyone else')
print(compare({'Uses prepaid': y, 'No bank account': d.unbanked == 1, 'Rents home': nn(d.renter, d.renter == 1),
    'Income under $40k': d.income <= 3, 'Under 35': d.age_under35 == 1,
    'Used check cashing, payday, pawn or money transfer': d.alt_financial_services_count >= 1,
    'Income varies month to month': nn(d.income_volatility, d.income_volatility >= 2),
    'Contacted by debt collector': nn(d.debt_collector_contact, d.debt_collector_contact == 1),
    'Savings under $1,000 (of those reporting)': nn(d.savings_range, d.savings_range <= 3),
    'Turned down for credit': nn(d.turned_down_for_credit, d.turned_down_for_credit == 1),
    'Lives in the South': d.region_south == 1}, TIER['BANK']).to_string())
print(f"Top 20% holds {y[TIER['BANK']].sum() / y.sum():.1%} of all users")
khlow = d.fin_knowledge_KH.rank(pct=True) <= 1 / 3
fwb = raw.FWBscore.where(raw.FWBscore >= 0)
print('\nQ3 CFPB: at-risk tier (top 30% of scores) vs everyone else')
print(compare({'Uses prepaid': y, 'Knowledge in bottom third': khlow, 'Low basic literacy (0-1 of 3)': d.fin_literacy_LM <= 1,
    'No bank account': d.unbanked == 1, "Couldn't raise $2,000": nn(d.can_raise_2000, d.can_raise_2000 <= 2),
    'Any material hardship': nn(d.hardship_count, d.hardship_count >= 1),
    'Felt mistreated (sometimes/often)': nn(d.felt_mistreated_by_fin_firms, d.felt_mistreated_by_fin_firms >= 3),
    'Contacted by debt collector': nn(d.debt_collector_contact, d.debt_collector_contact == 1),
    'Used payday loan': raw.PRODUSE_1 == 1}, TIER['CFPB']).to_string())
print(f"Tier holds {y[TIER['CFPB']].sum() / y.sum():.1%} of users; financial well-being {fwb[TIER['CFPB']].mean():.1f} vs {fwb[~TIER['CFPB']].mean():.1f}")
G = {'No bank account + low basic literacy': (d.unbanked == 1) & (d.fin_literacy_LM <= 1),
     'Debt collector + denied/discouraged from credit': (d.debt_collector_contact == 1) & (d.credit_constrained == 1),
     "Couldn't raise $2,000 + income varies": (d.can_raise_2000 <= 2) & (d.income_volatility >= 2),
     'Low basic literacy (0-1 of 3)': d.fin_literacy_LM <= 1, 'Not working (unemployed / unable)': d.emp_unemployed_or_unable == 1,
     'Felt mistreated by financial firms': d.felt_mistreated_by_fin_firms >= 3,
     'Used payday, pawn, check cashing or money transfer': d.alt_financial_services_count >= 1}
print('\nQ3 named groups')
print(pd.DataFrame({k: {'n': int(g.sum()), '% of adults': 100 * g.mean(), 'usage %': 100 * y[g].mean(),
      'x average': y[g].mean() / y.mean(), '% of all users': 100 * y[g].sum() / y.sum()} for k, g in G.items()}).T.round(1).to_string())
print(f"\nKnowledge gap: {khlow[y == 1].mean():.1%} of users vs {khlow[y == 0].mean():.1%} of non-users are in the bottom third")
print(f"Reported a problem to an agency: {(raw.CONSPROTECT3[y == 1] == 1).mean():.1%} of users vs {(raw.CONSPROTECT3[y == 0] == 1).mean():.1%} of non-users")
print('\nFairness check (race/ethnicity and sex are NOT model inputs)')
fair = {}
for k, lab in {1: 'White', 2: 'Black', 3: 'Other', 4: 'Hispanic'}.items():
    m = prot.PPETHM == k
    fair[lab] = {'usage %': 100 * y[m].mean(), 'in bank top 20% %': 100 * TIER['BANK'][m].mean(), 'in CFPB tier %': 100 * TIER['CFPB'][m].mean()}
for k, lab in {1: 'Male', 2: 'Female'}.items():
    m = prot.PPGENDER == k
    fair[lab] = {'usage %': 100 * y[m].mean(), 'in bank top 20% %': 100 * TIER['BANK'][m].mean(), 'in CFPB tier %': 100 * TIER['CFPB'][m].mean()}
print(pd.DataFrame(fair).T.round(1).to_string())